# Your own Colab copy

Choose File, Save a copy in Drive before working. Run the setup cell once, then work through the starter. Download your notebook and exported ZIP before disconnecting. Colab files can disappear when a runtime ends. Submit through the hub after finishing your attempt.

In [ ]:
from pathlib import Path
import hashlib
import os
import urllib.request
import zipfile

work = Path('/content/icaic-basketball') if Path('/content').exists() else Path.cwd() / 'icaic-basketball'
work.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', '1')
payload = urllib.request.urlopen('https://elteomc.github.io/icaic-practice-hub/packs/basketball.zip', timeout=90).read()
assert hashlib.sha256(payload).hexdigest() == '2e79df2133960489a3df23d17e58f763a9742841c75cdcf9ffb8b9fd932ea483', 'The task pack changed. Reopen the latest notebook from the hub.'
archive_path = work / 'task.zip'
archive_path.write_bytes(payload)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(work)
os.chdir(work / 'basketball')
import sys
sys.path.insert(0, str(Path.cwd()))
print('Task folder:', Path.cwd())


# Basketball Shooting

Read Statement.md before changing code. This starter checks the input and export path. It deliberately contains no trained solution. Keep your experiments and reasoning here.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
train = pd.read_csv('train.csv')
print(train.shape)
print(train.head())
print('Made-shot rate:', train.shot_made_flag.mean())
plt.scatter(train.loc_x, train.loc_y, c=train.shot_made_flag, s=2, alpha=.2)
plt.xlabel('loc_x')
plt.ylabel('loc_y')
plt.show()

## Your training experiment

Create a stratified development split. Train using only loc_x and loc_y. The supplied single linear layer is a permitted but untrained starting model. Choose an appropriate loss for raw logits, inspect your development accuracy, and make preprocessing part of the exported model.

In [ ]:
import torch
from submission_model import MyModel
torch.set_num_threads(2)
torch.manual_seed(2026)
model = MyModel()
x = torch.tensor(train[['loc_x', 'loc_y']].to_numpy(), dtype=torch.float32)
y = torch.tensor(train.shot_made_flag.to_numpy(), dtype=torch.float32).reshape(-1, 1)
print('Input shape:', x.shape, 'output shape:', model(x[:4]).shape)

# Add your development split and training here


## Export and submit

Run this after training. If you change the architecture, update submission_model.py to match before exporting. Only the class definition and trained state dictionary belong in the ZIP. The cell also works now to establish an untrained baseline.

In [ ]:
from pathlib import Path
import zipfile
model.eval()
torch.save(model.state_dict(), 'submission_dic.pth')
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ['submission_model.py', 'submission_dic.pth']:
        archive.write(name, name)
print('Saved submission.zip:', Path('submission.zip').stat().st_size, 'bytes')

## Keep evidence

Record your development split, seed, result, and one next hypothesis. Submit submission.zip. A successful upload checks the pipeline, while improved development performance checks whether your training helps.

## Download for the hub

Run this export cell to download your ZIP. Open Submit & notes on the task page and follow the available grading instructions. If you are practicing independently, finish your attempt before checking the final practice score.

In [ ]:
from pathlib import Path
assert Path('submission.zip').is_file(), 'Run the model export cell first'
output = 'submission.zip'
try:
    from google.colab import files
    files.download(output)
except ImportError:
    print('Upload this file:', Path(output).resolve())
